# **Tahap 0: Cofiguration**

In [93]:
# Install dulu library buat baca RSS (Biasanya udah ada di Kaggle, tapi buat jaga-jaga)
%pip install -q feedparser pandas

Note: you may need to restart the kernel to use updated packages.


In [94]:
import feedparser
import pandas as pd
import urllib.parse
import re
from datetime import datetime
import torch
from transformers import pipeline

import os
import numpy as np
import random
import transformers as tf

from sklearn.preprocessing import MaxAbsScaler

In [95]:
# Maksa GPU (CuDNN) di Kaggle biar ga ngacak hitungan
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(42)

def kunci_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

kunci_seed(42)

In [96]:
MARKET_POSITIVE_WORDS = [
    "naik", "menguat", "positif", "tumbuh", "meningkat", "optimis", "rebound", 
    "bullish", "stabil", "surplus", "rekor", "cuan", "laba", "akumulasi", "prospek"
]

MARKET_NEGATIVE_WORDS = [
    "turun", "melemah", "negatif", "anjlok", "koreksi", "tertekan", "inflasi", 
    "resesi", "krisis", "rugi", "bearish", "risiko", "ketidakpastian", "perang", 
    "konflik", "gagal", "defisit"
]

STOPWORDS_ID = {
    "dan", "yang", "di", "ke", "dari", "untuk", "dengan", "dalam", "hari", 
    "ini", "terbaru", "adalah", "pada", "karena", "sebagai", "itu", "akan", 
    "bisa", "ada", "tidak", "juga", "sudah", "saja", "lagi", "atau", "oleh",
    "untuk", "kita", "kami", "saya", "kamu", "mereka", "dia", "saat", "bagi"
}

# **TAHAP 1 NLP: Transplantasi Otak Sentimen.**

* **Mempertahankan Scraper Emas:** Kita tetep pakai `feedparser` dan `urllib.parse` dari `app.py` lu karena itu udah standar industri buat nyedot RSS Google News secara bersih dan terstruktur.
* **Injeksi HuggingFace Transformers:** Kita bakal manggil *library* `transformers` buat nge-*load* model `mdhugol/indonesia-bert-sentiment-classifier`. Karena lu lagi duduk manis di Kaggle dengan fasilitas Dual T4 GPU (VRAM 15GB), kita bakal nyuruh model ini langsung jalan di GPU (`device=0`) biar eksekusinya kilat.
* **Transisi ke Softmax Probability:** AI lu nggak akan lagi ngeluarin teks kaku "Positif" atau "Negatif". Model ini akan membaca kalimat utuh dan mengeluarkan probabilitas desimal dari *layer Softmax*. Contoh: *Headline* berita A memiliki probabilitas 80% Positif, 15% Netral, dan 5% Negatif. Angka-angka halus inilah yang sangat dibutuhkan oleh sistem *Fuzzy Logic* lu nanti.

In [97]:
NEWS_KEYWORDS = {
    "ANTM": ["ANTM saham", "Antam emas", "Aneka Tambang saham", "ANTM emas"],
    "MDKA": ["MDKA saham", "Merdeka Copper Gold saham", "MDKA emas"],
    "BRMS": ["BRMS saham", "Bumi Resources Minerals saham", "BRMS emas"],
    "PSAB": ["PSAB saham", "J Resources Asia Pasifik saham", "PSAB emas", "J Resources gold", "J Resources tambang emas"],
    "ACES": ["ACES saham", "Ace Hardware saham", "ACES retail"]
}

MARKET_KEYWORDS =  ["harga emas hari ini", "emas dunia", "emas antam", "harga emas naik", "harga emas turun", "gold price today", 
                    "IHSG hari ini", "saham Indonesia", "pasar modal Indonesia", "Bursa Efek Indonesia", "rekomendasi saham hari ini",
                    "ekonomi Indonesia", "inflasi Indonesia", "suku bunga Bank Indonesia", "rupiah hari ini", "pertumbuhan ekonomi Indonesia",
                    "ekonomi global", "The Fed suku bunga", "inflasi Amerika", "resesi global", "geopolitical risk market",
                    "harga komoditas", "harga minyak dunia", "harga batu bara", "harga tembaga", "commodity market",
                    "saham ramai dibahas", "market trend today", "investor sentiment", "fear of missing out saham", "berita ekonomi viral"
]

In [98]:
# ======================================================
# 1. LOAD MODEL INDOBERT (OTAK BARU)
# ======================================================
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
otak_nlp = pipeline(
    "text-classification", 
    model="mdhugol/indonesia-bert-sentiment-classification", 
    # tokenizer = AutoTokenizer.from_pretrained("mdhugol/indonesia-bert-sentiment-classification"),
    top_k=None, # Biar semua probabilitas (Pos, Neu, Neg) keluar
    device=0 
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mdhugol/indonesia-bert-sentiment-classification
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [99]:
def bersihkan_teks(teks):
    # 1. Lowercase: IndoBERT versi ini sensitif sama huruf kapital
    teks = teks.lower()
    
    # 2. Hapus URL/Link (kalau tiba-tiba ada nyelip di RSS)
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)
    
    # 3. Hapus entitas Twitter (Mention & Hashtag) karena modelnya dari data sosmed
    teks = re.sub(r'\@\w+|\#\w+', '', teks)
    
    # 4. Hapus karakter HTML (kayak &amp;, &quot;)
    teks = re.sub(r'&[a-z]+;', ' ', teks)
    
    # 5. Hapus tanda baca berlebihan dan karakter non-alfanumerik 
    # (Kita sisain angka karena buat saham angka 20% itu penting)
    teks = re.sub(r'[^\w\s]', ' ', teks)
    
    # 6. Hapus spasi ganda dari hasil pemotongan di atas
    teks = re.sub(r'\s+', ' ', teks).strip()
    
    return teks

In [100]:
def sedot_berita_expert(ticker, keywords_list, max_per_keyword=10):
    print(f"[*] (Scraping & Preprocessing) Memproses: {ticker}...")
    news_items = []
    
    for keyword in keywords_list:
        query = urllib.parse.quote(f"{keyword} when:30d")
        url = f"https://news.google.com/rss/search?q={query}&hl=id&gl=ID&ceid=ID:id"
        feed = feedparser.parse(url)
        
        for entry in feed.entries[:max_per_keyword]:
            judul_kotor = entry.title if "title" in entry else ""
            judul_bersih_portal = judul_kotor.rsplit(" - ", 1)[0] if " - " in judul_kotor else judul_kotor

            judul_siap_nlp = bersihkan_teks(judul_bersih_portal)
            
            # Lewatin kalau judulnya kosong setelah dibersihin
            if not judul_siap_nlp: 
                continue
                
            # label_index = {'LABEL_0': 'positive', 'LABEL_1': 'neutral', 'LABEL_2': 'negative'}
            # status = label_index[hasil_bert['label']]
            # score = hasil_bert['score']
            hasil_bert = otak_nlp(judul_siap_nlp)[0]
            prob_pos, prob_neu, prob_neg = 0.0, 0.0, 0.0
            
            for sentimen in hasil_bert:
                if sentimen['label'] == 'LABEL_0': prob_pos = sentimen['score']
                elif sentimen['label'] == 'LABEL_1': prob_neu = sentimen['score']
                elif sentimen['label'] == 'LABEL_2': prob_neg = sentimen['score']
                    
            news_items.append({
                "Ticker": ticker,
                "Keyword": keyword,
                "Tanggal": entry.published if "published" in entry else None,
                "Judul_Asli": judul_bersih_portal,
                "Judul_Preprocessed": judul_siap_nlp, # Biar lu bisa inspeksi bedanya
                "Prob_Positif": prob_pos,
                "Prob_Netral": prob_neu,
                "Prob_Negatif": prob_neg
            })
            # news_items.append({
            #     "Ticker": ticker,
            #     "Keyword": keyword,
            #     "Tanggal": entry.published if "published" in entry else None,
            #     "Judul_Asli": judul_bersih_portal,
            #     "Judul_Preprocessed": judul_siap_nlp, # Biar lu bisa inspeksi bedanya
            #     "kelas": status,
            #     "confidence": score
            # })
            
    df_news = pd.DataFrame(news_items)
    
    if not df_news.empty:
        df_news["Tanggal"] = pd.to_datetime(df_news["Tanggal"], errors="coerce").dt.date
        df_news = df_news.dropna(subset=["Judul_Asli"]).drop_duplicates(subset=["Judul_Asli"])
        
    return df_news

In [101]:
# ======================================================
# 3. EKSEKUSI MASSAL (Semua Emiten)
# ======================================================
lemari_sentimen = {}

for ticker, list_keyword in NEWS_KEYWORDS.items():
    lemari_sentimen[ticker] = sedot_berita_expert(ticker, list_keyword, max_per_keyword=15)

[*] (Scraping & Preprocessing) Memproses: ANTM...
[*] (Scraping & Preprocessing) Memproses: MDKA...
[*] (Scraping & Preprocessing) Memproses: BRMS...
[*] (Scraping & Preprocessing) Memproses: PSAB...
[*] (Scraping & Preprocessing) Memproses: ACES...


In [102]:
# Cek hasil buat salah satu saham, misal PSAB
pd.set_option('display.max_colwidth', None)
print("\n📊 CONTOH HASIL PANEN BERITA (PSAB):")
lemari_sentimen['PSAB'][['Keyword', 'Judul_Asli', 'Judul_Preprocessed', 'Prob_Positif', 'Prob_Negatif']].head(50)
# lemari_sentimen['PSAB'][['Keyword', 'Judul_Asli', 'Judul_Preprocessed', 'kelas', 'confidence']].head(50)


📊 CONTOH HASIL PANEN BERITA (PSAB):


,Keyword,Judul_Asli,Judul_Preprocessed,Prob_Positif,Prob_Negatif
0,PSAB saham,"Rekomendasi Saham Hari Ini: AMMN, BMRI, PGAS & PSAB, IHSG Diprediksi Rebound Terbatas",rekomendasi saham hari ini ammn bmri pgas psab ihsg diprediksi rebound terbatas,0.001965,0.000612
1,PSAB saham,"IHSG lanjutkan reli, saham emiten emas PSAB, ARCI dan BRMS ditutup melaju",ihsg lanjutkan reli saham emiten emas psab arci dan brms ditutup melaju,0.002105,0.000706
2,PSAB saham,PSAB Tambah Kendali Tambang Lewat Akuisisi Rp86 Miliar,psab tambah kendali tambang lewat akuisisi rp86 miliar,0.001934,0.000578
3,PSAB saham,"Gerak IHSG Kian Lincah, Serok Saham PSAB, HRTA, OASA, TOBA, dan ESSA",gerak ihsg kian lincah serok saham psab hrta oasa toba dan essa,0.003684,0.000686
4,PSAB saham,"Trump Effect: Saham Emiten Emas Melesat, Ada yang Naik Nyaris 13% - Market",trump effect saham emiten emas melesat ada yang naik nyaris 13 market,0.004785,0.001428
5,PSAB saham,"IHSG Lanjutkan Reli, Saham Emiten Emas PSAB, ARCI dan BRMS Ditutup Melaju",ihsg lanjutkan reli saham emiten emas psab arci dan brms ditutup melaju,0.002105,0.000706
6,PSAB saham,"IHSG Rawan Koreksi Lanjutan, 6 Saham Justru Bakal Cuan",ihsg rawan koreksi lanjutan 6 saham justru bakal cuan,0.000900,0.004354
7,PSAB saham,"10 Saham dengan Penurunan Jumlah Investor Terdalam April 2026, Ada Emiten Emas",10 saham dengan penurunan jumlah investor terdalam april 2026 ada emiten emas,0.002027,0.001004
8,PSAB saham,Harga Saham Tambang Emas Mulai Bangkit 10 April 2026,harga saham tambang emas mulai bangkit 10 april 2026,0.004045,0.000596
9,PSAB saham,Saham Emas Pulih di BEI Analis Prediksi Tren Positif Masih Berlanjut,saham emas pulih di bei analis prediksi tren positif masih berlanjut,0.003392,0.000567


# **2. Agregasi Harian & Kalibrasi Sentimen.**

* **Grouping Temporal (Penyelarasan Dimensi):** Model XGBoost lu yang kemaren itu murni nerima data harian (1 baris per hari). Kalau hari ini ada 15 berita tentang BRMS yang rilis, lu nggak mungkin masukin 15 baris ini ke sistem Fusi. Kita harus melebur (agregasi) 15 probabilitas tersebut menjadi 1 nilai rata-rata (*mean*) representatif untuk hari itu.
* **Koreksi Normalisasi (Softmax Calibration):** *Output* probabilitas mentah dari *pipeline* HuggingFace kadang terdistorsi kalau teksnya kurang panjang. Kita harus memaksa total probabilitas (Positif + Netral + Negatif) menjadi mutlak 1.0 (100%) secara matematis agar tidak ada bias distribusi.
* **Reduksi Dimensi (Skor Tunggal):** Sistem *Fuzzy Inference Mamdani* lu bakal mabok kalau dikasih 3 variabel probabilitas cuma buat merepresentasikan 1 sentimen. Kita harus kompres angka ini pakai rumus `(Prob_Positif - Prob_Negatif)`, sehingga ngasilin satu parameter tunggal dengan rentang -1 (Bearish/Negatif Parah) sampai +1 (Bullish/Positif Parah).

In [103]:
def agregasi_sentimen_harian(df_news):
    print(f"[*] Melakukan kompresi dan agregasi harian...")
    
    if df_news.empty:
        return pd.DataFrame()

    # 1. GROUPING BERDASARKAN TANGGAL
    # Mengambil rata-rata dari seluruh probabilitas berita di hari perdagangan yang sama
    df_harian = df_news.groupby(['Ticker', 'Tanggal']).agg(
        Total_Berita=('Judul_Asli', 'count'), # Ngitung berapa berita yang nge-backup sentimen hari ini
        Avg_Prob_Positif=('Prob_Positif', 'mean'),
        Avg_Prob_Netral=('Prob_Netral', 'mean'),
        Avg_Prob_Negatif=('Prob_Negatif', 'mean')
    ).reset_index()
    
    # 2. KOREKSI NORMALISASI ABSOLUT
    # Memastikan total probabilitas = 1.0 (100%) buat menghilangkan distorsi Softmax
    total_prob = df_harian['Avg_Prob_Positif'] + df_harian['Avg_Prob_Netral'] + df_harian['Avg_Prob_Negatif']
    
    # Pake np.where buat ngehindarin error division by zero (jaga-jaga kalau probnya 0 semua)
    df_harian['Avg_Prob_Positif'] = np.where(total_prob > 0, df_harian['Avg_Prob_Positif'] / total_prob, 0)
    df_harian['Avg_Prob_Netral'] = np.where(total_prob > 0, df_harian['Avg_Prob_Netral'] / total_prob, 0)
    df_harian['Avg_Prob_Negatif'] = np.where(total_prob > 0, df_harian['Avg_Prob_Negatif'] / total_prob, 0)
    
    # 3. PENCIPTAAN SKOR TUNGGAL (Amunisi buat Fuzzy Logic)
    # Rentang: -1 (Sangat Negatif) sampai +1 (Sangat Positif)
    df_harian['Skor_Sentimen_Final'] = df_harian['Avg_Prob_Positif'] - df_harian['Avg_Prob_Negatif']
    
    # Urutin dari tanggal terbaru ke terlama biar bacanya enak
    df_harian = df_harian.sort_values(by='Tanggal', ascending=False).reset_index(drop=True)
    
    return df_harian

In [106]:
# ======================================================
# EKSEKUSI AGREGASI BUAT SEMUA SAHAM
# ======================================================
lemari_sentimen_harian = {}

scaler_sentimen = MaxAbsScaler()

for ticker, df_mentah in lemari_sentimen.items():
    lemari_sentimen_harian[ticker] = agregasi_sentimen_harian(df_mentah)
    skor_mentah = lemari_sentimen_harian[ticker]['Skor_Sentimen_Final'].values.reshape(-1, 1)
    lemari_sentimen_harian[ticker]['Skor_Sentimen_Final'] = scaler_sentimen.fit_transform(skor_mentah)

[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...
[*] Melakukan kompresi dan agregasi harian...


In [107]:
# Kita inspeksi hasilnya buat PSAB
print("\n📊 HASIL AGREGASI HARIAN (PSAB):")
print(lemari_sentimen_harian['PSAB'])


📊 HASIL AGREGASI HARIAN (PSAB):
  Ticker     Tanggal  Total_Berita  Avg_Prob_Positif  Avg_Prob_Netral  \
0   PSAB  2026-05-22             6          0.001717         0.997344   
1   PSAB  2026-05-20             3          0.001229         0.992874   
2   PSAB  2026-05-19             1          0.001965         0.997422   
3   PSAB  2026-05-15             4          0.002702         0.996212   
4   PSAB  2026-05-12             2          0.002053         0.996587   
5   PSAB  2026-05-11             2          0.001794         0.997598   
6   PSAB  2026-05-08             3          0.077573         0.921409   
7   PSAB  2026-05-07             3          0.002623         0.995610   
8   PSAB  2026-05-06             5          0.002635         0.996328   
9   PSAB  2026-05-04             1          0.003347         0.995875   

   Avg_Prob_Negatif  Skor_Sentimen_Final  
0          0.000939             0.010169  
1          0.005897            -0.060979  
2          0.000612             0.

# **Tahap 3: Export Output NLP (.csv/.json)**
* **Fakta Teknis:** Sama kayak model deret waktu kemarin, hasil olahan skor sentimen dari IndoBERT ini harus lu ekspor jadi file dataset terpisah (misal `brms_sentiment_output.csv`). File inilah yang nantinya bakal dijajalin berdampingan dengan prediksi XGBoost di depan meja persidangan Fuzzy Logic Mamdani (Tahap Akhir).

In [108]:
# Bikin folder khusus biar rapi
os.makedirs('data_sentimen', exist_ok=True)

for ticker, df in lemari_sentimen_harian.items():
    if not df.empty:
        nama_file = f"data_sentimen/sentimen_{ticker}.csv"
        df.to_csv(nama_file, index=False)
        print(f"   [+] Tersimpan: {nama_file}")

   [+] Tersimpan: data_sentimen/sentimen_ANTM.csv
   [+] Tersimpan: data_sentimen/sentimen_MDKA.csv
   [+] Tersimpan: data_sentimen/sentimen_BRMS.csv
   [+] Tersimpan: data_sentimen/sentimen_PSAB.csv
   [+] Tersimpan: data_sentimen/sentimen_ACES.csv
